# BIOPINN — data generation + training (local)

Generates the synthetic FDM dataset via Latin Hypercube Sampling and trains the
BIOPINN physics-informed network, saving the processed dataset and the trained
checkpoint straight into this checkout's own `artifacts/` and `data/` folders.

This is the **local-only** variant of the training notebook: no Google Drive mount,
no Colab detection, no git clone -- it assumes you already have this repo checked out
on disk and are running Jupyter from inside it. If you need the Colab version instead
(e.g. to use a free T4 GPU without a local NVIDIA card), use
`notebooks/biopinn_train.ipynb`, which auto-detects Colab vs. local and handles both.

This notebook is a **thin orchestration layer**: every piece of science/ML logic
(the FDM solver, the microenvironment model, the PINN architecture, the loss
functions, the training loop) lives in the `src/` package and is imported here
unchanged. Nothing is reimplemented in this notebook -- local scripts
(`scripts/run_evaluation.py`, `scripts/run_dashboard.py`, ...) call the exact same
`src/` functions on the artifacts this notebook produces.

**Sections:** (1) setup &nbsp;(2) GPU check &nbsp;(3) configuration &nbsp;(4) generate
FDM data &nbsp;(5) train the PINN &nbsp;(6) save artifacts &nbsp;(7) quick sanity plots.

**Expected runtime:** with `QUICK_TEST = True` (default), the whole notebook runs in
a few minutes and is meant for verifying the pipeline end-to-end -- not a usable
trained model. Set `QUICK_TEST = False` to generate the full 2,000-simulation
dataset and train at production scale -- budget **~1-3.5 hours** for data generation
(parallelized across every available CPU core; some Latin-Hypercube-sampled
small-tumor/small-nanoparticle combinations need heavy CFL sub-stepping, see
`src/fdm_solver.py`) plus the configured Adam+L-BFGS training budget. A GPU (see
section 2) speeds up training itself substantially; data generation is CPU-only
regardless (it's solving a finite-difference PDE, not running the neural network).

> **Nothing here checkpoints partway through.** If this machine sleeps, loses power,
> or the kernel dies mid-run, that progress is lost. For a long `QUICK_TEST=False`
> run, keep the machine awake (disable sleep/hibernate) for the duration.


## 1. Setup

In [ ]:
import os
from pathlib import Path


def _find_local_repo_root(start: Path) -> Path:
    """Walk upward from `start` looking for the repo root (has setup.py + src/).
    Handles this notebook being run from notebooks/, the repo root, or anywhere
    Jupyter happens to set its working directory to, as long as it's somewhere
    inside the checkout."""
    for candidate in (start, *start.parents):
        if (candidate / "setup.py").exists() and (candidate / "src").is_dir():
            return candidate
    raise RuntimeError(
        f"Could not find the BIOPINN repo root by walking up from {start}. Make sure this "
        "notebook is inside the cloned/unzipped repo (e.g. biopinn/notebooks/biopinn_train_local.ipynb)."
    )


REPO_DIR = _find_local_repo_root(Path.cwd())
os.chdir(REPO_DIR)
print(f"Using this checkout at {REPO_DIR}")
print("cwd:", os.getcwd())


In [ ]:
# Editable install so `import src` resolves to this checkout, plus the pinned
# scientific-Python / PyTorch stack from requirements.txt. Safe to re-run --
# pip no-ops quickly if everything is already installed in the active environment.
!pip install -q -e .
!pip install -q -r requirements.txt


## 2. GPU check

Training runs on CPU if no GPU is detected -- fine for `QUICK_TEST=True`, slow for a
full-scale run. If this machine has an NVIDIA GPU and PyTorch doesn't see it, it's
almost always because the CPU-only build got installed (the default from plain PyPI
on every platform, not just Windows) instead of a CUDA build.


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
    if "+cpu" in torch.__version__:
        print(
            "\nNo GPU detected -- this environment has the CPU-only PyTorch build "
            f"({torch.__version__}). If this machine has an NVIDIA GPU:\n"
            "  1. Check your driver's max supported CUDA version with `nvidia-smi` in a terminal\n"
            "     (top-right corner, e.g. \"CUDA Version: 12.3\" -- that's a ceiling, not an exact match).\n"
            "  2. Reinstall PyTorch with a matching CUDA build in this same environment, e.g.:\n"
            "       pip uninstall -y torch\n"
            "       pip install torch --index-url https://download.pytorch.org/whl/cu121\n"
            "     (swap cu121 for whatever https://pytorch.org/get-started/locally/ shows for your\n"
            "     driver's CUDA version -- available tags change over time as new PyTorch releases\n"
            "     ship, so don't assume cu121 is still current; if a tag 404s, try the next one down\n"
            "     from what the selector suggests, e.g. cu124, cu126, cu128.)\n"
            "  3. RESTART THE KERNEL and re-run this notebook from the top.\n"
            "If this machine has no NVIDIA GPU, CPU is expected -- QUICK_TEST=True still runs "
            "in a few minutes; a full-scale run (QUICK_TEST=False) will just take substantially "
            "longer than on a GPU."
        )
    else:
        print("\nNo GPU detected -- training will run on CPU (fine for QUICK_TEST=True, slow for a full-scale run).")

print("Using device:", DEVICE)


In [ ]:
import src
print("biopinn src package version:", src.__version__)


## 3. Configuration

Outputs are written straight into this checkout's own `artifacts/` and `data/`
folders -- the repo-relative defaults already in `configs/default_config.yaml` --
exactly where `scripts/run_evaluation.py`, `scripts/run_dashboard.py`, etc. already
expect to find them. Nothing to redirect or download afterward.


In [ ]:
# Flip to False to generate the full production dataset (2,000 sims) and train at
# full scale (20,000 Adam iters + up to 5,000 L-BFGS iters). True runs the small
# experiment_1 dev config end-to-end in a few minutes, to sanity-check the whole
# pipeline before committing to a multi-hour production run.
QUICK_TEST = True
SEED = 42

from src.config import load_config, resolve_path

config = load_config("experiment_1" if QUICK_TEST else None)

# Resolved once, used by every "save this artifact" cell below.
ARTIFACTS_DIR = resolve_path(config, "artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print("experiment:", config["experiment_name"])
print("n_simulations:", config["dataset"]["split"]["train"] + config["dataset"]["split"]["val"] + config["dataset"]["split"]["test"])
print("device:", DEVICE)
print("artifacts will be saved under:", ARTIFACTS_DIR)

# Set to True to skip section 4 below and load a dataset already generated by
# scripts/generate_dataset.py (or a previous run of this notebook) from
# paths.processed instead of regenerating it. Useful after running the standalone
# script for real multi-core speed on Windows (see section 4's markdown), or to
# resume after generation finished but training was interrupted.
DATA_ALREADY_GENERATED = False


## 4. Generate the synthetic FDM dataset

Latin Hypercube samples the 5D parameter space (tumor radius, nanoparticle
diameter, surface concentration, decay rate, duration), solves the forward-Euler
FDM reference solution for each combination (`src/fdm_solver.py`, with the CFL
guard auto-refining the internal time step), samples data/collocation/BC/IC points
for the PINN losses, and writes normalized `train/val/test.npz` +
`normalization_stats.json`. See `src/data_pipeline.py::build_dataset`.

`n_jobs` parallelizes across CPU cores via `ProcessPoolExecutor` on Linux/Mac.
**Windows** can't safely do that from inside a Jupyter cell (each worker re-imports
the kernel launcher as `__main__` and fails to bootstrap, crashing with
`BrokenProcessPool`), so this cell handles that automatically instead: on Windows it
runs `scripts/generate_dataset.py` as a real subprocess (proper `__main__` guard,
parallelizes safely across every CPU core) and loads the result -- no manual steps
needed.


In [ ]:
import os
import subprocess
import sys
import time
from src.data_pipeline import build_dataset, load_processed_dataset

if DATA_ALREADY_GENERATED:
    print("Loading pre-generated dataset from", resolve_path(config, "processed"))
    dataset = load_processed_dataset(config)
    for split_name, tensors in dataset["splits"].items():
        print(f"  {split_name}: data_X {tensors['data_X'].shape}, collocation_X {tensors['collocation_X'].shape}")
elif sys.platform == "win32":
    # Windows' spawn-based multiprocessing can't parallelize ProcessPoolExecutor work
    # safely from inside a Jupyter cell (each worker re-imports the kernel launcher as
    # "__main__" and fails to bootstrap, crashing with BrokenProcessPool). Running the
    # standalone script as a real subprocess sidesteps this entirely -- it has a proper
    # `if __name__ == "__main__":` guard, so it parallelizes correctly across every CPU
    # core. Its output lands exactly where this notebook's `config` already expects, so
    # the load_processed_dataset call below picks it straight up.
    experiment_args = ["--experiment", "experiment_1"] if QUICK_TEST else []
    print("Windows detected -- generating via scripts/generate_dataset.py for real multi-core speed...")
    subprocess.run(
        [sys.executable, "scripts/generate_dataset.py", *experiment_args, "--seed", str(SEED)],
        check=True,
    )
    dataset = load_processed_dataset(config)
    for split_name, tensors in dataset["splits"].items():
        print(f"  {split_name}: data_X {tensors['data_X'].shape}, collocation_X {tensors['collocation_X'].shape}")
else:
    n_jobs = os.cpu_count() or 1
    print(f"Solving FDM simulations across {n_jobs} worker process(es)...")

    t0 = time.time()
    dataset = build_dataset(config, seed=SEED, save=True, n_jobs=n_jobs)
    elapsed = time.time() - t0

    print(f"\nDataset generation complete in {elapsed/60:.1f} min.")
    for split_name, tensors in dataset["splits"].items():
        print(f"  {split_name}: {len(dataset['sims'][split_name])} sims, "
              f"data_X {tensors['data_X'].shape}, collocation_X {tensors['collocation_X'].shape}")


## 5. Build & train the BIOPINN network

Two-phase training (`src/train.py::train`): Adam with StepLR decay, gradient
clipping, and a w_phys warmup ramp / NaN-recovery safeguard, followed by L-BFGS
fine-tuning with strong-Wolfe line search. Stops early once the validation physics
residual and data loss both stay under their configured thresholds for enough
consecutive epochs. Saves the best-validation checkpoint + normalization stats
into `artifacts/`.


**If training crashes with `CUDA out of memory`:** training is full-batch (every iteration sees the whole train split at once), and the physics loss's second-order autograd over millions of collocation points can exceed GPU memory at full production scale. `configs/default_config.yaml`'s `training.max_points_per_chunk` (default 1,000,000) already caps this by computing each loss term in point-count-bounded chunks -- mathematically identical gradient, bounded peak memory. If you still OOM, lower it (e.g. 500,000 or 250,000); if you have VRAM to spare and want marginally less looping overhead, raise it or remove the key entirely.

In [ ]:
from src.train import train

t0 = time.time()
result = train(config, dataset, device=DEVICE, save=True)
elapsed = time.time() - t0

print(f"\nTraining complete in {elapsed/60:.1f} min "
      f"({result['adam_epochs_run']} Adam epochs, "
      f"{result['lbfgs_closure_evaluations']} L-BFGS closure evaluations).")
print(f"Final validation data loss: {result['final_val_data']:.4e}")
print(f"Final validation physics residual: {result['final_val_phys']:.4e}")


## 6. Save artifacts

The processed dataset and the model checkpoint were already written directly to
`artifacts/` / `data/processed/` in steps 4-5 -- nothing left to copy. This cell
just confirms the files landed and additionally saves the loss history + the exact
config used, for reproducibility and for the sanity plots below.


In [ ]:
import json

artifact_paths = result["artifacts"]
print("checkpoint:", artifact_paths["checkpoint_path"], "(",
      os.path.getsize(artifact_paths["checkpoint_path"]) / 1e6, "MB )")
print("normalization stats:", artifact_paths["normalization_stats_path"])

for split_name in ("train", "val", "test"):
    npz_path = os.path.join(config["paths"]["processed"], f"{split_name}.npz")
    print(split_name, "->", npz_path, "(", os.path.getsize(npz_path) / 1e6, "MB )")


In [ ]:
# Loss history + the exact resolved config, alongside the checkpoint, so a later
# session can reproduce the run's plots without re-training.
run_record_path = ARTIFACTS_DIR / "training_run.json"
with open(run_record_path, "w", encoding="utf-8") as f:
    json.dump({
        "config": config,
        "history": result["history"],
        "adam_epochs_run": result["adam_epochs_run"],
        "lbfgs_closure_evaluations": result["lbfgs_closure_evaluations"],
        "final_val_data": result["final_val_data"],
        "final_val_phys": result["final_val_phys"],
    }, f, indent=2)
print("saved run record:", run_record_path)

# Also save just the loss history under the exact filename Phase 14's
# results-generation pipeline expects (src/results.py looks for this first,
# falling back to extracting "history" from training_run.json above).
history_path = resolve_path(config, "training_history")  # respects paths.training_history (e.g. experiment_1's "_dev" suffix)
with open(history_path, "w", encoding="utf-8") as f:
    json.dump(result["history"], f, indent=2)
print("saved training history:", history_path)


## 7. Quick sanity plots

Loss curves per component, and a predicted-vs-FDM-reference concentration profile
for one held-out validation simulation, so an obviously broken run is caught here
rather than three phases later in `scripts/run_evaluation.py`.


In [ ]:
import matplotlib.pyplot as plt

history = result["history"]
adam_n = result["adam_epochs_run"]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, key in zip(axes.ravel(), ("total", "data", "phys", "bc", "neu", "ic")):
    vals = history[key]
    if any(v > 0 for v in vals):
        ax.semilogy(vals)
    else:
        ax.plot(vals)  # identically-zero series (e.g. ic, via the hard-IC transform)
    ax.axvline(adam_n, color="gray", linestyle="--", linewidth=1)
    ax.set_title(f"{key} loss")
    ax.set_xlabel("iteration")
    ax.grid(alpha=0.3)
plt.suptitle(f"BIOPINN training loss curves ({config['experiment_name']})")
plt.tight_layout()
fig_path = ARTIFACTS_DIR / "training_loss_curves.png"
plt.savefig(fig_path, dpi=150)
print("saved:", fig_path)
plt.show()


In [ ]:
import json
import numpy as np
import torch

from src.data_pipeline import PARAM_ORDER, _solve_one
from src.model import BIOPINN

# Pick one validation-split simulation and compare the trained model's prediction
# against its own FDM reference solution.
if "sims" in dataset:
    val_sim = dataset["sims"]["val"][0]
else:
    # DATA_ALREADY_GENERATED=True: load_processed_dataset doesn't keep the raw
    # (r, t, C) arrays in memory, so re-solve one val-split simulation on demand
    # from the exact parameters saved in sim_params.json -- the same approach
    # src/evaluate.py uses to reconstruct the test split's reference fields.
    processed_dir = resolve_path(config, "processed")
    with open(processed_dir / "sim_params.json", encoding="utf-8") as f:
        val_params = json.load(f)["val"][0]
    val_sim = _solve_one((val_params["sim_id"], np.array([val_params[k] for k in PARAM_ORDER]), config))

r, t, C_fdm = val_sim["r"], val_sim["t"], val_sim["C"]
stats = dataset["stats"]

def normalize_param(value, key):
    lo, hi = stats[key]["min"], stats[key]["max"]
    return 0.0 if hi <= lo else (value - lo) / (hi - lo)

param_row = [normalize_param(val_sim[k], k) for k in ("R_um", "d_NP_nm", "C0_uM", "k_d_per_hr", "t_max_hr")]

model = result["model"].to("cpu").eval()
t_snapshot = val_sim["t_max_hr"]  # final time
t_idx = np.argmin(np.abs(t - t_snapshot))

r_norm = torch.tensor(r / val_sim["R_um"], dtype=torch.float32).reshape(-1, 1)
t_norm = torch.full_like(r_norm, t[t_idx] / val_sim["t_max_hr"])
param_cols = torch.tensor(param_row, dtype=torch.float32).repeat(len(r), 1)
X = torch.cat([r_norm, t_norm, param_cols], dim=1)

with torch.no_grad():
    C_pred = (model(X).numpy().ravel()) * val_sim["C0_uM"]

plt.figure(figsize=(7, 5))
plt.plot(r, C_fdm[t_idx, :], "k-", linewidth=2, label="FDM reference")
plt.plot(r, C_pred, "r--", linewidth=2, label="PINN prediction")
plt.xlabel("radius (um)")
plt.ylabel("concentration (uM)")
plt.title(f"Validation sim {val_sim['sim_id']} at t={t[t_idx]:.1f}hr "
          f"(R={val_sim['R_um']:.0f}um, d_NP={val_sim['d_NP_nm']:.0f}nm)")
plt.legend()
plt.grid(alpha=0.3)
fig_path = ARTIFACTS_DIR / "validation_sanity_check.png"
plt.savefig(fig_path, dpi=150)
print("saved:", fig_path)
plt.show()


## Next steps

Nothing to move -- steps 4-6 above already wrote everything directly into this
checkout's `artifacts/` and `data/processed/`. From here on (CPU is enough --
nothing below here re-trains):

```bash
python scripts/run_evaluation.py    # six-metric report + H1/H2/H4 pass/fail
python scripts/run_ablation.py      # physics-informed vs. unconstrained baseline
python scripts/run_optimization.py  # optimal (d_NP*, C0*) per tumor radius
python scripts/make_figures.py      # publication figures
python scripts/run_dashboard.py     # interactive results dashboard
python scripts/generate_results.py  # manuscript figures + tables -> results/paper/
```
